In [1]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console

In [10]:
model = OpenAIChatCompletionClient(model="gpt-4o-mini")

# 명확성 agent
clarity_agent = AssistantAgent(
    "ClarityAgent",
    model_client=model,
    system_message="""명확성과 간결함을 최우선으로 하는 전문 편집자로서, 저는 질문에 대한 내용으로만 편집을하며 모호함과 중복을 제거하여 모든 문장을 군더더기 없이 명쾌하게 만드는 역할을 수행합니다.
    설득이나 감정적인 어조에 신경 쓰기보다, 메시지를 누구나 쉽고 빠르게 읽고 이해할 수 있도록 만드는 데 집중합니다.    
    """,
)

# 톤(어조) agent : 이메일의 톤을 개선
tone_agent = AssistantAgent(
    "ToneAgent",
    model_client=model,
    system_message="""커뮤니케이션 코치로서, 저는 이메일의 정중함과 전문성을 유지하면서도 따뜻하고 자신감 넘치는 인간미가 느껴지도록 다듬어 드리는 역할을 맡습니다. 
    감정적인 공감대를 높이고 문장을 세련되게 다듬으며, 딱딱하거나 차갑게 느껴지는 표현 혹은 지나치게 가벼운 어조를 상황에 맞게 조정해 드릴 것입니다.
    """,
)

# 설득 agent : 이메일을 더 설득력 있게
persuasion_agent = AssistantAgent(
    "PersuasionAgent",
    model_client=model,
    system_message="""당신은 마케팅, 행동 심리학, 카피라이팅 교육을 받은 설득 전문가입니다. 당신의 임무는 이메일의 설득력을 높이는 것입니다. 
    구체적으로는 행동 유도(CTA) 개선, 논리 구조 설계, 그리고 혜택(Benefits)을 강조하는 일을 수행합니다. 또한, 힘이 없고 수동적인 표현은 모두 제거하십시오.
    """,
)

# 합성 agent : 모든 아이디를 받아서 이메일에 넣음
synthesizer_agent = AssistantAgent(
    "SynthesizerAgent",
    model_client=model,
    system_message="""당신은 고급 이메일 작성 전문가입니다. 
    당신의 역할은 이전 에이전트들의 모든 응답과 수정 사항을 검토한 뒤, 최상의 아이디어들을 종합하여 통일되고 세련된 이메일 초안을 작성하는 것입니다. 
    다음 사항에 집중해 주세요: 명확성, 어조, 설득력 개선 사항의 통합; 일관성, 유창함, 그리고 자연스러운 목소리 확보; 전문적이고 효과적이며 가독성이 좋은 버전 생성.
    """,
)

# 최종 결과
critic_agent = AssistantAgent(
    "CriticAgent",
    model_client=model,
    system_message="""당신은 이메일 품질 평가자입니다. 
    당신의 임무는 종합된 이메일에 대한 최종 검토를 수행하고 전문적인 기준을 충족하는지 판단하는 것입니다. 
    다음 사항을 검토하십시오: 명확성과 흐름, 적절하고 전문적인 어조, 효과적인 행동 유도, 그리고 전반적인 일관성. 
    건설적이되 단호하게 평가하십시오. 
    이메일에 중대한 결함(불분명한 메시지, 비전문적인 어조 또는 핵심 요소 누락)이 있는 경우, 한 가지 구체적인 개선 제안을 제공하십시오. 
    이메일이 전문적인 기준을 충족하고 효과적으로 전달된다면, 'The email meets professional standards.'라고 응답한 뒤 다음 줄에 TERMINATE를 적으십시오. 
    완벽하지 않더라도 전문적인 용도로 사용하기에 충분히 훌륭하다면 이메일을 승인해야 합니다.
    """,
)

In [4]:


text_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=30)
termination_condition = text_termination | max_messages_termination

In [11]:
team = RoundRobinGroupChat(
    participants=[
        clarity_agent,
        tone_agent,
        persuasion_agent,
        synthesizer_agent,
        critic_agent
    ],
    termination_condition=termination_condition
)

await Console(
    team.run_stream(task="안녕하세요. 프로젝트가 거의 끝났으니 검토해주세요.")
)

---------- TextMessage (user) ----------
안녕하세요. 프로젝트가 거의 끝났으니 검토해주세요.
---------- TextMessage (ClarityAgent) ----------
안녕하세요. 프로젝트 검토를 시작하겠습니다. 관련 문서를 제공해주시면 자료를 확인하겠습니다.
---------- TextMessage (ToneAgent) ----------
안녕하세요. 프로젝트가 거의 완료되었습니다. 검토를 부탁드리며, 관련 문서들을 함께 첨부하였습니다. 확인 후 의견 주시면 감사하겠습니다.
---------- TextMessage (PersuasionAgent) ----------
안녕하세요. 프로젝트가 거의 완료되었습니다! 검토해주시면 감사하겠습니다. 첨부한 관련 문서들을 확인해주시고, 피드백을 주시면 더욱 발전할 수 있습니다. 여러분의 의견이 중요합니다!
---------- TextMessage (SynthesizerAgent) ----------
제목: 프로젝트 검토 요청

안녕하세요,

프로젝트가 거의 완료되었습니다. 검토를 부탁드리며, 관련 문서들을 첨부하였습니다. 여러분의 소중한 의견이 저희에게 큰 도움이 될 것입니다. 확인 후 피드백을 주시면 감사하겠습니다.

감사합니다.

[당신의 이름]  
[당신의 직책]  
[당신의 연락처]  
[회사명]
---------- TextMessage (CriticAgent) ----------
이 이메일은 전체적으로 명확성과 흐름이 좋고, 전문적인 어조를 유지하고 있습니다. 요청이 명시되어 있으며 피드백을 요청하는 부분도 효과적으로 전달되었습니다. 다만, 이메일 본문의 첫 부분에서 "검토를 부탁드리며, 관련 문서들을 첨부하였습니다."라는 문장은 약간 약하게 들릴 수 있습니다. 좀 더 직접적인 요청으로 개선하면 좋겠습니다.

구체적인 개선 제안: "프로젝트 검토를 부탁드리며, 첨부한 관련 문서들을 꼭 확인해주시기 바랍니다."로 문장을 변경하여 보다 명확한 행동 유도를 생성할 수 있습니

TaskResult(messages=[TextMessage(id='ba929f2a-ee7b-4528-8958-722eca08a928', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 1, 30, 13, 5, 58, 259613, tzinfo=datetime.timezone.utc), content='안녕하세요. 프로젝트가 거의 끝났으니 검토해주세요.', type='TextMessage'), TextMessage(id='b6ae0c9a-bb1b-41e5-a2c9-722e15b75270', source='ClarityAgent', models_usage=RequestUsage(prompt_tokens=120, completion_tokens=24), metadata={}, created_at=datetime.datetime(2026, 1, 30, 13, 5, 59, 465640, tzinfo=datetime.timezone.utc), content='안녕하세요. 프로젝트 검토를 시작하겠습니다. 관련 문서를 제공해주시면 자료를 확인하겠습니다.', type='TextMessage'), TextMessage(id='25900671-18a3-4317-9b0b-3d8bb9ca08bd', source='ToneAgent', models_usage=RequestUsage(prompt_tokens=161, completion_tokens=37), metadata={}, created_at=datetime.datetime(2026, 1, 30, 13, 6, 1, 73432, tzinfo=datetime.timezone.utc), content='안녕하세요. 프로젝트가 거의 완료되었습니다. 검토를 부탁드리며, 관련 문서들을 함께 첨부하였습니다. 확인 후 의견 주시면 감사하겠습니다.', type='TextMessage'), TextMessage(id='b67d13cc-6935-472d